<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/19_end_to_end_rag_system/rag_ui_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U gradio sentence-transformers transformers scikit-learn faiss-cpu sentencepiece --quiet

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
documents = [
    "Paris is the capital of France.",
    "France is located in Europe.",
    "Berlin is the capital of Germany.",
    "Python is a programming language."
]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

embeddings = embed_model.encode(documents)

In [14]:
def rag_pipeline(user_query):
    # Step 1: Rewrite query (STRICT)
    rewrite_prompt = f"""
Convert the following into a proper question.

Query: {user_query}

Only return the question.
"""
    inputs = tokenizer(rewrite_prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=30, do_sample=False)
    query = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Step 2: Retrieval
    query_embedding = embed_model.encode([query])
    query_tfidf = vectorizer.transform([query])

    semantic_scores = cosine_similarity(query_embedding, embeddings)[0]
    keyword_scores = cosine_similarity(query_tfidf, tfidf_matrix)[0]

    hybrid_scores = 0.5 * semantic_scores + 0.5 * keyword_scores

    best_index = hybrid_scores.argmax()
    context = documents[best_index]

    # 🔥 DEBUG PRINT (IMPORTANT)
    print("QUERY:", query)
    print("CONTEXT:", context)

    # Step 3: Answer generation (VERY STRICT)
    prompt = f"""
Answer the question using ONLY the context below.

Context: {context}

Question: {query}

Answer:
"""
    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("ANSWER:", answer)

    return answer

In [15]:
import gradio as gr

interface = gr.Interface(
    fn=rag_pipeline,
    inputs="text",
    outputs="text",
    title="RAG Question Answering System",
    description="Ask any question and get an answer using RAG pipeline."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://40ccd1350f3cce1f37.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
